# **Project Name** - Amazon Prime TV Shows and Movies: Regression and Classification

##### **Project Type** - Regression / Classification
##### **Contribution** - Individual
##### **Team Member 1 -** Add your name
##### **Team Member 2 -** N/A
##### **Team Member 3 -** N/A
##### **Team Member 4 -** N/A

# **Project Summary -**

This project analyses the supplied Amazon Prime TV Shows and Movies dataset, which is provided as a merged credits-and-titles CSV. The file contains 124,235 credit-level records and 19 columns, including person and title identifiers, cast/crew information, content type, release year, runtime, genres, production countries, seasons, and IMDb/TMDb measures. Because the same title can appear for multiple credited people, title-level analysis and modelling first remove repeated records using the title identifier (`id`). The unique-title dataset is the correct unit for counting and prediction.

The work has two modelling objectives. The regression task estimates a title's IMDb score using available metadata. The classification task predicts whether a title is a movie or a show. These targets are directly present in the supplied file and represent distinct supervised-learning problems. The regression model uses numerical metadata and selected categorical descriptors; the classification model uses title metadata while excluding identifiers and fields that directly reveal the target. The dataset does not include a verified Amazon Prime availability flag, so the analysis does not claim that every record is currently available on Prime Video.

The workflow includes data inspection, duplicate and missing-value analysis, data cleaning, feature construction from genre and country lists, visual exploration, statistical hypothesis tests, preprocessing, train/test splitting, model training, evaluation, cross-validation, hyperparameter tuning, and model persistence. Missing numerical values are imputed using medians and categorical values using the most frequent category within scikit-learn pipelines. Categorical variables are one-hot encoded. The modelling workflow keeps preprocessing inside the pipeline to reduce data leakage.

For regression, MAE, RMSE, and R² are reported. For classification, accuracy, precision, recall, F1-score, and a confusion matrix are reported. Baseline and tuned Random Forest models are compared so that model performance is judged on unseen test data rather than training scores alone. The final outcome is a reproducible notebook that can be rerun with the supplied CSV. All numerical findings and model scores are generated when the notebook is executed; no performance values are assumed in advance.

# **GitHub Link -**

Add your GitHub repository URL after uploading the completed notebook and supporting files.

# **Problem Statement**


Using the supplied merged credits-and-titles dataset, explore title-level catalogue patterns and build two supervised learning models: (1) a regression model to estimate IMDb score and (2) a classification model to identify whether a title is a MOVIE or SHOW. Clean repeated credit rows, handle missing values, engineer useful metadata features, evaluate both models on a held-out test set, and explain the results. The CSV does not contain a verified Prime Video availability field; therefore platform availability is outside the scope of this analysis.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import ast, re, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix)
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (10, 5)

### Dataset Loading

In [ ]:
# Works in Jupyter and Google Colab. Keep the CSV beside this notebook when possible.
DATA_FILE = "merged_credits_titles.csv"
candidate_paths = [Path(DATA_FILE), Path("/mnt/data") / DATA_FILE, Path("/content") / DATA_FILE,
                   Path("/content/drive/MyDrive") / DATA_FILE]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        print("Upload merged_credits_titles.csv")
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("Dataset not found. Put merged_credits_titles.csv in the notebook folder.")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", DATA_PATH, "| Shape:", df_raw.shape)
display(df_raw.head())

### Dataset First View

In [ ]:
display(df_raw.head(10))
display(df_raw.sample(min(5, len(df_raw)), random_state=RANDOM_STATE))

### Dataset Rows & Columns count

In [ ]:
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")

### Dataset Information

In [ ]:
df_raw.info()

#### Duplicate Values

In [ ]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Duplicate title-credit identifiers (id + person_id + role):",
      df_raw.duplicated(subset=["id", "person_id", "role"]).sum())

#### Missing Values/Null Values

In [ ]:
missing = pd.DataFrame({"missing_count": df_raw.isna().sum(),
                         "missing_percent": (df_raw.isna().mean()*100).round(2)})
display(missing.sort_values("missing_percent", ascending=False))

In [ ]:
plt.figure(figsize=(12,5))
sns.heatmap(df_raw.isna(), cbar=False, yticklabels=False)
plt.title("Missing-value pattern (all source rows)")
plt.xlabel("Dataset columns")
plt.tight_layout()
plt.show()

### What did you know about your dataset?

The file contains 124,235 credit-level rows and 19 columns. Multiple rows can refer to the same title because each person-credit relationship is represented separately. The `id` column is therefore used to create a unique title-level dataset. The dataset includes both movies and shows, release and runtime information, genres, production countries, age certification, IMDb ratings/votes, and TMDb popularity/score. Several metadata fields contain missing values, which are examined before modelling.

## ***2. Understanding Your Variables***

In [ ]:
print(df_raw.columns.tolist())

In [ ]:
display(df_raw.describe(include="all").T)

### Variables Description

`id` identifies a title; `person_id` identifies a credited person. `title`, `type`, `description`, `release_year`, `runtime`, `genres`, `production_countries`, `age_certification`, and `seasons` describe the content. `imdb_score` and `imdb_votes` describe IMDb audience ratings and vote volume. `tmdb_popularity` and `tmdb_score` are TMDb measures. `name`, `character`, and `role` describe credited people and their contribution. Credit-related columns repeat across a title's cast/crew rows, so they are not treated as independent title observations.

### Check Unique Values for each variable.

In [ ]:
for col in df_raw.columns:
    print(f"\n{col}: {df_raw[col].nunique(dropna=True)} unique values")
    if df_raw[col].nunique(dropna=True) <= 15:
        print(df_raw[col].value_counts(dropna=False).head(15))

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
def parse_list(value):
    if pd.isna(value) or value == "":
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

# One row per title: source file is at person-credit granularity.
df = df_raw.drop_duplicates(subset="id").copy()
for col in ["genres", "production_countries"]:
    df[col + "_list"] = df[col].apply(parse_list)
df["main_genre"] = df["genres_list"].str[0].fillna("Unknown")
df["main_country"] = df["production_countries_list"].str[0].fillna("Unknown")
df["genre_count"] = df["genres_list"].str.len()
df["country_count"] = df["production_countries_list"].str.len()
df["type"] = df["type"].astype("string").str.upper().str.strip()
print("Rows before title deduplication:", len(df_raw))
print("Unique title rows after deduplication:", len(df))
display(df.head())

### What all manipulations have you done and insights you found?

Removed repeated title-credit rows by keeping one record per `id`; standardized the content type; converted list-like genre and country strings into lists; derived primary genre/country and their counts; and prepared numerical and categorical fields. Rows with a missing target are retained for general EDA but excluded from the corresponding supervised model. These steps prevent inflated title counts and provide compact, interpretable metadata features.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='type', order=df['type'].value_counts().index); plt.title('Number of titles by content type'); plt.xlabel('Content type'); plt.ylabel('Unique titles')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine content type distribution and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 2

In [ ]:
plt.figure(figsize=(10,5))
df['release_year'].dropna().astype(int).value_counts().sort_index().plot(); plt.title('Titles released by year'); plt.xlabel('Release year'); plt.ylabel('Unique titles')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine titles released by year and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 3

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['imdb_score'].dropna(), bins=25, kde=True); plt.title('IMDb score distribution'); plt.xlabel('IMDb score')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine imdb score distribution and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 4

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='type', y='runtime', showfliers=False); plt.title('Runtime by content type'); plt.ylabel('Runtime (minutes)')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine runtime distribution by type and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 5

In [ ]:
plt.figure(figsize=(10,5))
df['genres_list'].explode().value_counts().head(15).sort_values().plot(kind='barh'); plt.title('Top 15 genres'); plt.xlabel('Title count')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine top genres and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 6

In [ ]:
plt.figure(figsize=(10,5))
df['production_countries_list'].explode().value_counts().head(15).sort_values().plot(kind='barh'); plt.title('Top 15 production countries'); plt.xlabel('Title count')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine top production countries and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 7

In [ ]:
plt.figure(figsize=(10,5))
df['age_certification'].fillna('Not rated').value_counts().head(12).sort_values().plot(kind='barh'); plt.title('Age certification counts'); plt.xlabel('Title count')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine age certification and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 8

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='type', y='imdb_score'); plt.title('IMDb score by content type'); plt.ylabel('IMDb score')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine imdb score by type and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 9

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(np.log1p(df['imdb_votes'].dropna()), bins=30, kde=True); plt.title('Log-transformed IMDb votes distribution'); plt.xlabel('log(1 + IMDb votes)')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine imdb votes distribution and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 10

In [ ]:
plt.figure(figsize=(10,5))
((df['release_year']//10)*10).value_counts().sort_index().plot(kind='bar'); plt.title('Titles by release decade'); plt.xlabel('Release decade'); plt.ylabel('Unique titles')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine release decade and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 11

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df.sample(min(3000,len(df)),random_state=42), x='tmdb_popularity', y='imdb_score', hue='type', alpha=.5); plt.title('TMDb popularity vs IMDb score')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine tmdb popularity vs imdb score and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 12

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df[df['type']=='SHOW'], y='seasons'); plt.title('Seasons among TV shows'); plt.ylabel('Number of seasons')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine seasons by content type and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 13

In [ ]:
plt.figure(figsize=(10,5))
genre_rows=df[['imdb_score','genres_list']].explode('genres_list').dropna(); top=genre_rows['genres_list'].value_counts().head(12).index; genre_rows[genre_rows['genres_list'].isin(top)].groupby('genres_list')['imdb_score'].mean().sort_values().plot(kind='barh'); plt.title('Average IMDb score by common genre'); plt.xlabel('Mean IMDb score')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to examine mean imdb score by genre and identify distributions or differences across the title catalogue.

##### 2. What is/are the insight(s) found from the chart?

Use the plotted values to identify the leading categories, overall pattern, spread, and any unusual observations. The chart is descriptive and does not establish causation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

These findings can guide catalogue segmentation, content discovery, and further investigation of audience preferences. Decisions should be validated with business, licensing, and platform-availability data.

#### Chart - 14 - Correlation Heatmap

In [ ]:
numeric = df.select_dtypes(include=np.number).copy()
plt.figure(figsize=(11,8))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap of numeric variables")
plt.tight_layout()
plt.show()

A correlation heatmap summarizes pairwise linear relationships among numerical variables and helps identify potential redundancy.

See the corresponding preprocessing and model outputs above.

Inspect the displayed coefficients: values near +1 indicate positive association, values near -1 indicate negative association, and values near 0 indicate weak linear association. Correlation does not imply causation.

See the corresponding analysis and printed output above.

#### Chart - 15 - Pair Plot

In [ ]:
pair_cols = [c for c in ['release_year','runtime','imdb_score','imdb_votes','tmdb_popularity'] if c in df]
pair_data = df[pair_cols].dropna().sample(min(700, len(df[pair_cols].dropna())), random_state=42)
sns.pairplot(pair_data, corner=True, diag_kind='hist')
plt.show()

A pair plot was selected to compare distributions and pairwise relationships among important numerical variables in one view.

See the corresponding preprocessing and model outputs above.

The plots show distributions on the diagonal and pairwise scatter patterns elsewhere. Use them to spot skew, clusters, and possible relationships; the sample is used to keep rendering efficient.

See the corresponding analysis and printed output above.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Three hypotheses are tested at a 5% significance level: (1) mean IMDb score differs between movies and shows; (2) mean runtime differs between movies and shows; (3) IMDb score is associated with IMDb vote count. The tests use title-level records and omit missing values only for the variables involved.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: The mean IMDb score is equal for movies and shows. H1: The mean IMDb score differs between movies and shows.

#### 2. Perform an appropriate statistical test.

In [ ]:
movie_scores = df.loc[df['type']=='MOVIE','imdb_score'].dropna()
show_scores = df.loc[df['type']=='SHOW','imdb_score'].dropna()
test1 = stats.ttest_ind(movie_scores, show_scores, equal_var=False, nan_policy='omit')
print(f"Movie mean={movie_scores.mean():.3f}; Show mean={show_scores.mean():.3f}")
print(f"Welch t={test1.statistic:.4f}, p-value={test1.pvalue:.6g}")
print("Decision:", "Reject H0" if test1.pvalue < 0.05 else "Fail to reject H0")

##### Which statistical test have you done to obtain P-Value?

See the corresponding preprocessing and model outputs above.

Welch's independent two-sample t-test compares the means of two independent groups and does not assume equal variances.

A Welch t-test is appropriate because the target is numeric and the two content types are independent groups; Welch's version is robust to unequal group variances.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: The mean runtime is equal for movies and shows. H1: The mean runtime differs between movies and shows.

#### 2. Perform an appropriate statistical test.

In [ ]:
movie_runtime = df.loc[df['type']=='MOVIE','runtime'].dropna()
show_runtime = df.loc[df['type']=='SHOW','runtime'].dropna()
test2 = stats.ttest_ind(movie_runtime, show_runtime, equal_var=False, nan_policy='omit')
print(f"Movie mean runtime={movie_runtime.mean():.2f}; Show mean runtime={show_runtime.mean():.2f}")
print(f"Welch t={test2.statistic:.4f}, p-value={test2.pvalue:.6g}")
print("Decision:", "Reject H0" if test2.pvalue < 0.05 else "Fail to reject H0")

##### Which statistical test have you done to obtain P-Value?

See the corresponding preprocessing and model outputs above.

Welch's independent two-sample t-test compares average runtime between the two independent content-type groups without requiring equal variances.

See the corresponding preprocessing and model outputs above.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: There is no monotonic association between IMDb votes and IMDb score (Spearman rho = 0). H1: A monotonic association exists (Spearman rho is not 0).

#### 2. Perform an appropriate statistical test.

In [ ]:
vote_score = df[['imdb_votes','imdb_score']].dropna()
test3 = stats.spearmanr(vote_score['imdb_votes'], vote_score['imdb_score'])
print(f"Spearman rho={test3.statistic:.4f}, p-value={test3.pvalue:.6g}")
print("Decision:", "Reject H0" if test3.pvalue < 0.05 else "Fail to reject H0")

##### Which statistical test have you done to obtain P-Value?

See the corresponding preprocessing and model outputs above.

Spearman correlation is suitable for assessing a monotonic relationship between two numeric variables and is less sensitive to non-normal distributions and extreme values than Pearson correlation.

See the corresponding preprocessing and model outputs above.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Modelling-specific imputation is performed inside sklearn pipelines to prevent leakage.
num_cols = ['release_year','runtime','seasons','imdb_votes','tmdb_popularity','genre_count','country_count']
cat_cols = ['age_certification','main_genre','main_country']
display(df[num_cols + cat_cols].isna().sum().sort_values(ascending=False).to_frame('missing_count'))
print("Median numerical imputation and most-frequent categorical imputation will be fitted on training data only.")

#### What all missing value imputation techniques have you used and why did you use those techniques?

For descriptive analysis, numerical missing values are not automatically filled because missingness itself is informative and some fields are not applicable (for example, seasons for movies). For modelling, `SimpleImputer(strategy='median')` is used for numerical predictors and `most_frequent` for categorical predictors inside each pipeline. Target rows with missing IMDb score are excluded only from regression; rows with missing type are excluded from classification.

### 2. Handling Outliers

In [ ]:
outlier_summary = []
for col in ['runtime','imdb_votes','tmdb_popularity','imdb_score']:
    s = df[col].dropna()
    q1, q3 = s.quantile([.25,.75]); iqr=q3-q1
    low, high=q1-1.5*iqr, q3+1.5*iqr
    outlier_summary.append([col, low, high, ((s<low)|(s>high)).sum()])
display(pd.DataFrame(outlier_summary, columns=['feature','IQR_lower_bound','IQR_upper_bound','potential_outliers']))
print("Potential extremes are retained unless verified as data errors; they may be genuine titles/popularity values.")

##### What all outlier treatment techniques have you used and why did you use those techniques?

IQR bounds are calculated for inspection. Extreme runtime, vote, and popularity values may be genuine content characteristics, so they are not blindly deleted. The Random Forest models are tree-based and do not require outlier scaling; the notebook reports distributions and uses robust median imputation. Any business decision to cap/remove outliers should be validated with the project mentor.

### 3. Categorical Encoding

In [ ]:
# Demonstration of nominal category encoding; actual encoding is fitted in the model pipelines.
display(pd.get_dummies(df[['type','age_certification']].fillna('Unknown').head(8), dtype=int))

#### What all categorical encoding techniques have you used & why did you use those techniques?

One-hot encoding is used for categorical predictors because categories such as type, genre, country, and certification have no natural numeric order. `handle_unknown='ignore'` allows the fitted pipeline to process categories not seen during training. Numerical fields are median-imputed; no ordinal encoding is imposed on nominal categories.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# No description-text modelling is required for these structured metadata tasks.
def expand_contractions(text):
    return text
print("Text contraction expansion skipped: free text is not used as a predictor.")

#### 2. Lower Casing

In [ ]:
print("Lowercasing skipped: no text feature is used in the two selected models.")

#### 3. Removing Punctuations

In [ ]:
print("Punctuation removal skipped: no text feature is used in the two selected models.")

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
print("URL/digit cleaning skipped: no description text is used in modelling.")

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
print("Stopword removal skipped: no NLP feature is used in modelling.")

In [ ]:
print("Whitespace cleaning skipped: no NLP feature is used in modelling.")

#### 6. Rephrase Text

In [ ]:
print("Text rephrasing is not applicable to structured metadata modelling.")

#### 7. Tokenization

In [ ]:
print("Tokenization is not applicable because the description column is excluded.")

#### 8. Text Normalization

In [ ]:
print("Stemming/lemmatization is not applied; text is not part of the feature set.")

##### Which text normalization technique have you used and why?

The supplied description field is not required for the two metadata-based models. Text cleaning, stop-word removal, tokenization, POS tagging, lemmatization, and TF-IDF are therefore not applied. This avoids adding an unnecessary NLP pipeline and keeps the model focused on structured metadata.

#### 9. Part of speech tagging

In [ ]:
print("POS tagging is not applicable because text modelling is outside this project's scope.")

#### 10. Text Vectorization

In [ ]:
print("TF-IDF/CountVectorizer is not applied; categorical metadata is one-hot encoded instead.")

##### Which text vectorization technique have you used and why?

Text vectorization is not used because the selected regression and classification tasks are built from structured metadata rather than free-text descriptions. A separate NLP experiment could use TF-IDF on descriptions in future work.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Feature engineering already created main_genre, main_country, genre_count and country_count.
engineered_cols = ['main_genre','main_country','genre_count','country_count']
display(df[engineered_cols].head())

#### 2. Feature Selection

In [ ]:
# Targets and predictors are separated independently for the two tasks.
regression_target = 'imdb_score'
classification_target = 'type'
print("Regression target:", regression_target)
print("Classification target:", classification_target)
print("Exclude IDs, names, characters, descriptions, and target columns from predictors.")

Feature selection is based on domain relevance and leakage prevention. Identifiers, person names, character names, and title names are excluded. For regression, TMDb score is excluded because it is another rating measure that could act as a near-proxy for IMDb score. For classification, `type` is the target and is never included among predictors.

See the corresponding preprocessing and model outputs above.

##### Which all features you found important and why?

Useful predictors include release year, runtime, seasons, IMDb votes, TMDb popularity, genre count, country count, primary genre, primary production country, and age certification. Their importance is measured after training; importance is model-dependent and should not be interpreted as causation.

### 5. Data Transformation

No target transformation is applied. Numerical predictors are median-imputed; scaling is included in the regression preprocessing pipeline for consistency, although Random Forest models do not require scaling. The target IMDb score remains on its original rating scale for clear interpretation.

In [ ]:
print("No target transformation is applied. Numerical features are imputed and scaled in the preprocessing pipeline.")

### 6. Data Scaling

In [ ]:
print("StandardScaler is applied to numeric predictors inside each pipeline. Tree models do not require scaling, but the pipeline is reproducible.")

StandardScaler is included for numerical predictors in the preprocessing pipeline. Tree-based models are largely insensitive to feature scale, but using a consistent preprocessing pipeline supports reproducibility and makes it straightforward to compare with linear models later.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

PCA is not required for this baseline project. The selected feature set is moderate in size and tree-based models can handle one-hot encoded features. Keeping the original features also makes model interpretation easier.

In [ ]:
print("PCA is not applied: the feature set is manageable and tree models support sparse one-hot features.")

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

No dimensionality reduction is applied. It is not necessary for the selected tree-based algorithms and would make the feature-level explanation less direct.

### 8. Data Splitting

In [ ]:
print("Both tasks use an 80/20 train-test split. Classification uses stratification to preserve class proportions.")

##### What data splitting ratio have you used and why?

An 80:20 train-test split is used. The model is trained on 80% of the title records and evaluated once on the remaining 20%. A fixed random seed makes the split reproducible. Classification uses stratification to preserve the movie/show proportions.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

The classification target is imbalanced: movies are much more frequent than shows in the supplied credit rows and remain the majority class at title level. Accuracy alone can therefore be misleading. The notebook reports per-class precision, recall, F1-score, and a confusion matrix in addition to accuracy.

In [ ]:
print("Class distribution for title-level classification target:")
display(df['type'].value_counts(dropna=False).to_frame('count'))
display(df['type'].value_counts(normalize=True).mul(100).round(2).to_frame('percent'))

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Class weighting (`class_weight='balanced'`) is used in the Random Forest classifier so errors on the less frequent class receive greater weight. We also report macro-F1 and per-class metrics; oversampling is not applied to the test data.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# REGRESSION: predict IMDb score from metadata.
reg_features = ['release_year','runtime','seasons','imdb_votes','tmdb_popularity',
                'genre_count','country_count','age_certification','main_genre','main_country']
reg_data = df.dropna(subset=['imdb_score']).copy()
X_reg = reg_data[reg_features]
y_reg = reg_data['imdb_score'].astype(float)
reg_num = ['release_year','runtime','seasons','imdb_votes','tmdb_popularity','genre_count','country_count']
reg_cat = ['age_certification','main_genre','main_country']
reg_pre = ColumnTransformer([
    ('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]), reg_num),
    ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), reg_cat)
])
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg,y_reg,test_size=.2,random_state=RANDOM_STATE)
reg_model = Pipeline([('preprocess',reg_pre),('model',RandomForestRegressor(n_estimators=200,min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1))])
reg_model.fit(Xr_train,yr_train)
yr_pred = reg_model.predict(Xr_test)
reg_metrics = {'MAE':mean_absolute_error(yr_test,yr_pred),'RMSE':mean_squared_error(yr_test,yr_pred)**.5,'R2':r2_score(yr_test,yr_pred)}
print("Regression test metrics:",reg_metrics)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(x=list(reg_metrics.keys()),y=list(reg_metrics.values()))
plt.title("Regression model evaluation metrics")
plt.ylabel("Score (different metric scales)")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6,5))
plt.scatter(yr_test,yr_pred,alpha=.35)
plt.plot([yr_test.min(),yr_test.max()],[yr_test.min(),yr_test.max()],'r--')
plt.xlabel("Actual IMDb score"); plt.ylabel("Predicted IMDb score")
plt.title("Regression: actual vs predicted")
plt.tight_layout(); plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning with randomized search and 3-fold CV.
reg_search = RandomizedSearchCV(
    reg_model,
    {'model__n_estimators':[150,250,350], 'model__max_depth':[None,15,25],
     'model__min_samples_split':[2,5,10], 'model__min_samples_leaf':[1,2,4],
     'model__max_features':[0.7,1.0]},
    n_iter=8, scoring='neg_root_mean_squared_error',
    cv=KFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,n_jobs=-1,verbose=1
)
reg_search.fit(Xr_train,yr_train)
best_reg_model=reg_search.best_estimator_
best_yr_pred=best_reg_model.predict(Xr_test)
best_reg_metrics={'MAE':mean_absolute_error(yr_test,best_yr_pred),
                  'RMSE':mean_squared_error(yr_test,best_yr_pred)**.5,
                  'R2':r2_score(yr_test,best_yr_pred)}
print("Best parameters:",reg_search.best_params_)
print("Untuned:",reg_metrics)
print("Tuned:",best_reg_metrics)

##### Which hyperparameter optimization technique have you used and why?

RandomizedSearchCV with stratified cross-validation is used to explore a compact range of Random Forest hyperparameters efficiently. It is more economical than an exhaustive grid when several parameters are considered.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The tuned model is compared with the untuned model using the same held-out test set. Improvement is determined from the actual MAE/RMSE/R² results printed by the notebook; lower MAE/RMSE and higher R² indicate better regression performance.

### ML Model - 2 (Classification: MOVIE vs SHOW)

#### 1. Explain the classification model used and its performance using evaluation metric score chart.

In [ ]:
clf_compare=pd.DataFrame([clf_metrics,best_clf_metrics],index=['Random Forest','Tuned Random Forest'])
display(clf_compare)
clf_compare.plot(kind='bar',figsize=(9,4),title='Classification: before vs after tuning')
plt.ylabel('Metric score'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
ConfusionMatrixDisplay.from_predictions(yc_test,best_yc_pred,normalize=None,xticks_rotation='vertical')
plt.title('Tuned classifier confusion matrix')
plt.tight_layout(); plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# CLASSIFICATION: predict whether a title is a MOVIE or SHOW.
clf_features = ['release_year','runtime','seasons','imdb_votes','tmdb_popularity',
                'genre_count','country_count','age_certification','main_genre','main_country']
clf_data=df.dropna(subset=['type']).copy()
X_clf=clf_data[clf_features]
y_clf=clf_data['type'].astype(str)
clf_num=['release_year','runtime','seasons','imdb_votes','tmdb_popularity','genre_count','country_count']
clf_cat=['age_certification','main_genre','main_country']
clf_pre=ColumnTransformer([
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),clf_num),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),clf_cat)
])
Xc_train,Xc_test,yc_train,yc_test=train_test_split(
    X_clf,y_clf,test_size=.2,random_state=RANDOM_STATE,stratify=y_clf)
clf_model=Pipeline([('preprocess',clf_pre),('model',RandomForestClassifier(
    n_estimators=200,class_weight='balanced',min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1))])
clf_model.fit(Xc_train,yc_train)
yc_pred=clf_model.predict(Xc_test)
clf_metrics={'Accuracy':accuracy_score(yc_test,yc_pred),
             'Precision_macro':precision_score(yc_test,yc_pred,average='macro',zero_division=0),
             'Recall_macro':recall_score(yc_test,yc_pred,average='macro',zero_division=0),
             'F1_macro':f1_score(yc_test,yc_pred,average='macro',zero_division=0)}
print("Classification test metrics:",clf_metrics)
print(classification_report(yc_test,yc_pred,zero_division=0))

# Tune classifier using macro-F1 to account for class imbalance.
clf_search=RandomizedSearchCV(
    clf_model,
    {'model__n_estimators':[150,250,350], 'model__max_depth':[None,15,25],
     'model__min_samples_split':[2,5,10], 'model__min_samples_leaf':[1,2,4],
     'model__max_features':['sqrt',0.7]},
    n_iter=8, scoring='f1_macro',
    cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,n_jobs=-1,verbose=1
)
clf_search.fit(Xc_train,yc_train)
best_clf_model=clf_search.best_estimator_
best_yc_pred=best_clf_model.predict(Xc_test)
best_clf_metrics={'Accuracy':accuracy_score(yc_test,best_yc_pred),
                  'Precision_macro':precision_score(yc_test,best_yc_pred,average='macro',zero_division=0),
                  'Recall_macro':recall_score(yc_test,best_yc_pred,average='macro',zero_division=0),
                  'F1_macro':f1_score(yc_test,best_yc_pred,average='macro',zero_division=0)}
print("Best classifier parameters:",clf_search.best_params_)
print("Tuned classifier metrics:",best_clf_metrics)
print(classification_report(yc_test,best_yc_pred,zero_division=0))


##### Which hyperparameter optimization technique have you used and why?

RandomizedSearchCV with stratified cross-validation optimizes macro-F1, which gives equal weight to the movie and show classes and is more informative than accuracy for an imbalanced target.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The tuned classifier is compared with the untuned classifier on the same test set. Macro-F1, per-class recall, and the confusion matrix are reviewed to ensure the minority SHOW class is not ignored.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

For regression, MAE is the average absolute rating error in IMDb points, RMSE penalizes larger errors, and R² measures explained variance. For classification, accuracy measures total correctness, while precision, recall, and F1 measure class-specific prediction quality. Macro-F1 treats MOVIE and SHOW equally and is important because the target classes are imbalanced. These outputs support catalogue metadata analysis but do not establish Prime availability or causation.

### ML Model - 3

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

In [ ]:
# Model 3 is not required: the assignment specifies exactly two models
# (one regression and one classification). Keep this section explicitly not applicable.
print("Model 3 not implemented: project requirement is two models only.")

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

In [ ]:
print("No third-model evaluation chart: only the required regression and classification models are implemented.")

#### 2. Cross- Validation & Hyperparameter Tuning

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

In [ ]:
print("No third-model tuning: only two models are required.")

##### Which hyperparameter optimization technique have you used and why?

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

See the corresponding preprocessing and model outputs above.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

See the corresponding preprocessing and model outputs above.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

For the regression task, prioritize low MAE and RMSE with a useful R². For classification, prioritize macro-F1 and SHOW-class recall alongside accuracy, because a high overall accuracy can hide poor performance on the minority class.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

The final regression and classification models are selected separately using held-out test metrics and cross-validation. The notebook prints the selected tuned model and its scores. Do not choose a model solely from training performance.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

Permutation importance on the held-out test set is used to estimate how much each feature contributes to model performance. It is a model-agnostic explanation, but correlated predictors can share or distort importance and the result does not prove causation.

**Note:** This section is covered by the two required models (regression and classification); a third model is not part of the stated requirement.

In [ ]:
# Permutation importance for both tasks, computed on held-out test samples.
reg_imp = permutation_importance(best_reg_model, Xr_test, yr_test, n_repeats=5,
                                 random_state=RANDOM_STATE, scoring='neg_mean_absolute_error', n_jobs=-1)
reg_imp_df = pd.DataFrame({'feature': reg_features, 'importance': reg_imp.importances_mean})
display(reg_imp_df.sort_values('importance', ascending=False))
reg_imp_df.sort_values('importance').plot(x='feature', y='importance', kind='barh',
    legend=False, figsize=(8,5), title='Regression permutation importance')
plt.tight_layout(); plt.show()

clf_imp = permutation_importance(best_clf_model, Xc_test, yc_test, n_repeats=5,
                                 random_state=RANDOM_STATE, scoring='f1_macro', n_jobs=-1)
clf_imp_df = pd.DataFrame({'feature': clf_features, 'importance': clf_imp.importances_mean})
display(clf_imp_df.sort_values('importance', ascending=False))
clf_imp_df.sort_values('importance').plot(x='feature', y='importance', kind='barh',
    legend=False, figsize=(8,5), title='Classification permutation importance')
plt.tight_layout(); plt.show()

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
MODEL_DIR=Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(best_reg_model,MODEL_DIR/"imdb_score_regression.joblib")
joblib.dump(best_clf_model,MODEL_DIR/"content_type_classification.joblib")
print("Saved models to:",MODEL_DIR.resolve())

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
loaded_reg=joblib.load(MODEL_DIR/"imdb_score_regression.joblib")
loaded_clf=joblib.load(MODEL_DIR/"content_type_classification.joblib")
print("Regression sample predictions:",loaded_reg.predict(Xr_test.head(3)))
print("Classification sample predictions:",loaded_clf.predict(Xc_test.head(3)))

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project uses the supplied merged credits-and-titles data to perform title-level preprocessing and develop two supervised models. IMDb score is treated as a regression target, while content type (MOVIE/SHOW) is treated as a classification target. The notebook compares baseline and tuned Random Forest pipelines, evaluates them with appropriate metrics, and saves both final pipelines. Results must be interpreted as estimates from the available historical metadata. The dataset does not include a verified Amazon Prime availability indicator, so conclusions are not claims about current Prime Video catalogue availability.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***